# CCK gold-chunk coverage: lines8_gold_newrepos
Records whether the chunk holding each answer was in the dense index, per cap. Runs benchmark/confound_sweep.py at pinned SHA 5a83318. Internet ON, CPU.


In [ ]:
!pip install fastembed transformers numpy --quiet
print('deps installed')


In [ ]:
from fastembed import TextEmbedding
from transformers import AutoTokenizer
MODEL = 'BAAI/bge-small-en-v1.5'
supported = [m['model'] for m in TextEmbedding.list_supported_models()]
assert MODEL in supported, f'{MODEL} not supported by this fastembed; supported: {supported}'
TextEmbedding(MODEL)
try:
    AutoTokenizer.from_pretrained(MODEL)
except Exception as e:
    print('tokenizer load failed (chunk length stats will be skipped):', str(e)[:200])
AutoTokenizer.from_pretrained('unsloth/gemma-2-2b-bnb-4bit')
print('all cached', flush=True)


In [ ]:
import subprocess, glob
SHA = '5a833188c90effa5298dcb4962704d80ad3700d6'
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(c, '->', r.returncode, r.stderr.strip()[-200:])
sh('git clone https://github.com/pupa3066/context-configuration-builder.git /tmp/cck')
sh(f'cd /tmp/cck && git checkout {SHA}')
for name, url in [('flask', 'https://github.com/pallets/flask.git'), ('httpx', 'https://github.com/encode/httpx.git'), ('rich', 'https://github.com/Textualize/rich.git')]:
    sh(f'git clone --depth=200 {url} /tmp/{name}')
    head = subprocess.run(f'git -C /tmp/{name} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
    print(name, len(glob.glob(f'/tmp/{name}/**/*.py', recursive=True)), '.py files, HEAD', head)


In [ ]:
import subprocess, sys, time
def stream(cmd):
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if 'SyntaxWarning' in line or line.strip().startswith('<unknown>'): continue
            print(line, end='', flush=True)
    return p.returncode
base = f'cd /tmp/cck && {sys.executable} -u benchmark/confound_sweep.py --repos flask:/tmp/flask httpx:/tmp/httpx rich:/tmp/rich --chunker lines8 --tag gold-newrepos --caps 1000 3000 100000'
print('--- SMOKE', time.strftime('%H:%M:%S'), flush=True)
smoke = base.replace('--caps 1000 3000 100000', '--caps 200')
rc = stream(smoke + ' --n-per-repo 3 --bootstrap 50 --out-dir /tmp/smoke')
assert rc == 0, 'smoke test failed - stop here and send the error'
print('--- REAL', time.strftime('%H:%M:%S'), flush=True)
rc = stream(base + ' --n-per-repo 80 --bootstrap 1000 --out-dir /kaggle/working')
print('done exit', rc, time.strftime('%H:%M:%S'))
